# ВНИМАНИЕ! ТУТ МОГУТ БЫТЬ КРИТИЧЕСКИЕ ОШИБКИ! НЕ ИСПОЛЬЗОВАТЬ КАК ТОРГОВУЮ МОДЕЛЬ

In [ ]:
from google.colab import drive
drive.mount('/content/gdrive')

Mounted at /content/gdrive


Теперь обучим модель random forest и сделаем предсказания.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd

df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/full_dataet_sentiment_prices.csv')
df = df.dropna()

# 2. ГРУППИРОВКА ПО ВРЕМЕНИ (Самый важный шаг!)
# Мы собираем все новости за одну свечу в одну строку
df_grouped = df.groupby('timestamp').agg({
    'open': 'first',      # Цена открытия свечи (она одна)
    'high': 'max',        # Максимум за это время
    'low': 'min',         # Минимум за это время
    'close': 'last',      # Цена закрытия (она одна)
    'volume': 'last',     # Объем (если он накопительный) или 'sum'
    'NEGATIVE': 'mean',   # Средний негатив за эту свечу
    'NEUTRAL': 'mean',    # Средний нейтрал
    'POSITIVE': 'mean'    # Средний позитив
}).sort_index() # Сортируем по времени, чтобы shift работал правильно

# Теперь df_grouped - это чистый временной ряд (одна строка = одна свеча)
df = df_grouped

# df = df.drop('Unnamed: 0', axis=1)
df['price_increased'] = (df['close'].shift(-1) > df['close']).astype(int)
df['NEGATIVE'] = df['NEGATIVE'].shift(1)
df['NEUTRAL'] = df['NEUTRAL'].shift(1)
df['POSITIVE'] = df['POSITIVE'].shift(1)
df['open_prev'] = df['open'].shift(1).pct_change() #
df['high_prev'] = df['high'].shift(1).pct_change() #
df['low_prev'] = df['low'].shift(1).pct_change()
df['volume_rvol'] = df['volume'] / df['volume'].rolling(20).mean() # rolling - берем предыдущие 20
df['close_prev'] = df['close'].shift(1).pct_change() #
df['open'] = df['open'].pct_change() #
df['volume_prev_rvol'] = df['volume_rvol'].shift(1)
df = df.dropna()

# 1. Сначала делим на Train и Test строго по времени (без перемешивания!)
# Берем последние 20% данных для теста (это наше "будущее")
train_size = int(len(df) * 0.8)
df_train = df.iloc[:train_size].copy()
df_test = df.iloc[train_size:].copy()

# 2. Балансируем ТОЛЬКО обучающую выборку (Train)
# Чтобы модель научилась видеть редкие классы (рост)
fall_train = df_train[df_train['price_increased'] == 0]
rise_train = df_train[df_train['price_increased'] == 1]

# Undersampling: обрезаем больший класс до размера меньшего
min_len = min(len(fall_train), len(rise_train))
df_train_balanced = pd.concat([
    fall_train.sample(n=min_len, random_state=42),
    rise_train.sample(n=min_len, random_state=42)
])

# Перемешиваем только train (чтобы деревья учились лучше)
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42)

# 3. Формируем X и y
columns = ['NEGATIVE', 'NEUTRAL', 'POSITIVE', 'open',
           'close_prev', 'high_prev', 'low_prev', 'volume_prev_rvol']

X_train = df_train_balanced[columns]
y_train = df_train_balanced['price_increased']

X_test = df_test[columns]  # Тест остался нетронутым!
y_test = df_test['price_increased']

print(f"Обучаем на: {len(X_train)} примерах (сбалансировано)")
print(f"Тестируем на: {len(X_test)} примерах (реальный рынок)")

Обучаем на: 119600 примерах (сбалансировано)
Тестируем на: 29979 примерах (реальный рынок)


In [ ]:
from numpy.random.mtrand import rand
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

# Инициализируем модель логистической регрессии
# Используем solver='liblinear' как хороший общий выбор для небольших и средних наборов данных
# random_state для воспроизводимости
# model = RandomForestClassifier(random_state=42)
model = LogisticRegression(solver='liblinear', random_state=42)

# Обучаем модель на обучающих данных
model.fit(X_train, y_train)

# Делаем предсказания на тестовых данных
# predict_proba возвращает вероятности принадлежности к каждому классу ([вероятность 0, вероятность 1])
y_pred_proba = model.predict_proba(X_test)[:, 1] # Берем вероятности для класса 1

# Преобразуем вероятности в бинарные предсказания, используя порог (например, 0.5)
y_pred_binary = (y_pred_proba > 0.5).astype(int)

print("Пример непрерывных предсказаний (вероятностей класса 1):", y_pred_proba[:10])
print("Пример бинарных предсказаний (с порогом 0.5):", y_pred_binary[:10])

Пример непрерывных предсказаний (вероятностей класса 1): [0.50199781 0.501801   0.49511779 0.50157907 0.50110919 0.50198718
 0.50123147 0.4954012  0.50275131 0.50134425]
Пример бинарных предсказаний (с порогом 0.5): [1 1 0 1 1 1 1 0 1 1]


Оценим производительность модели, используя метрики для классификации, так как целевая переменная бинарная.

In [ ]:
y_final_pred = []
# Попробуем более мягкий порог, чтобы поймать сделки
threshold_high = 0.5
threshold_low = 0.5

for p in y_pred_proba:
    if p > threshold_high:
        y_final_pred.append(1)
    elif p < threshold_low:
        y_final_pred.append(0)
    else:
        y_final_pred.append(2) # Пропуск хода

y_final_pred = np.array(y_final_pred)

# Фильтруем и считаем метрики
mask = y_final_pred != 2
if mask.sum() > 0:
    acc = accuracy_score(y_test[mask], y_final_pred[mask])
    print(f"\n=== РЕЗУЛЬТАТЫ СТРАТЕГИИ (Порог {threshold_high}/{threshold_low}) ===")
    print(f"Совершено сделок: {mask.sum()} из {len(y_test)} (Coverage: {mask.sum()/len(y_test)*100:.2f}%)")
    print(f"Точность (Win Rate): {acc:.4f}")
    print("\n", classification_report(y_test[mask], y_final_pred[mask]))
else:
    print("Модель не сделала ни одной ставки. Снизь порог уверенности.")


=== РЕЗУЛЬТАТЫ СТРАТЕГИИ (Порог 0.5/0.5) ===
Совершено сделок: 29979 из 29979 (Coverage: 100.00%)
Точность (Win Rate): 0.4963

               precision    recall  f1-score   support

           0       0.51      0.33      0.40     15296
           1       0.49      0.67      0.57     14683

    accuracy                           0.50     29979
   macro avg       0.50      0.50      0.48     29979
weighted avg       0.50      0.50      0.48     29979



# РАНДОМ ФОРЕСТ

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
import numpy as np
import pandas as pd

df = pd.read_csv('/content/gdrive/MyDrive/Colab Notebooks/course/datasets/full_dataet_sentiment_prices.csv')
df = df.dropna()

# 2. ГРУППИРОВКА ПО ВРЕМЕНИ (Самый важный шаг!)
# Мы собираем все новости за одну свечу в одну строку
df_grouped = df.groupby('timestamp').agg({
    'open': 'first',      # Цена открытия свечи (она одна)
    'high': 'max',        # Максимум за это время
    'low': 'min',         # Минимум за это время
    'close': 'last',      # Цена закрытия (она одна)
    'volume': 'last',     # Объем (если он накопительный) или 'sum'
    'NEGATIVE': 'mean',   # Средний негатив за эту свечу
    'NEUTRAL': 'mean',    # Средний нейтрал
    'POSITIVE': 'mean'    # Средний позитив
}).sort_index() # Сортируем по времени, чтобы shift работал правильно

# Теперь df_grouped - это чистый временной ряд (одна строка = одна свеча)
df = df_grouped

# df = df.drop('Unnamed: 0', axis=1)
df['price_increased'] = (df['close'].shift(-1) > df['close']).astype(int)
df['NEGATIVE'] = df['NEGATIVE'].shift(1)
df['NEUTRAL'] = df['NEUTRAL'].shift(1)
df['POSITIVE'] = df['POSITIVE'].shift(1)
df['open_prev'] = df['open'].shift(1).pct_change() #
df['high_prev'] = df['high'].shift(1).pct_change() #
df['low_prev'] = df['low'].shift(1).pct_change()
df['volume_rvol'] = df['volume'] / df['volume'].rolling(20).mean() # rolling - берем предыдущие 20
df['close_prev'] = df['close'].shift(1).pct_change() #
df['open'] = df['open'].pct_change() #
df['volume_prev_rvol'] = df['volume_rvol'].shift(1)
df = df.dropna()

# 1. Сначала делим на Train и Test строго по времени (без перемешивания!)
# Берем последние 20% данных для теста (это наше "будущее")
train_size = int(len(df) * 0.8)
df_train = df.iloc[:train_size].copy()
df_test = df.iloc[train_size:].copy()

# 2. Балансируем ТОЛЬКО обучающую выборку (Train)
# Чтобы модель научилась видеть редкие классы (рост)
fall_train = df_train[df_train['price_increased'] == 0]
rise_train = df_train[df_train['price_increased'] == 1]

# Undersampling: обрезаем больший класс до размера меньшего
min_len = min(len(fall_train), len(rise_train))
df_train_balanced = pd.concat([
    fall_train.sample(n=min_len, random_state=42),
    rise_train.sample(n=min_len, random_state=42)
])

# Перемешиваем только train (чтобы деревья учились лучше)
df_train_balanced = df_train_balanced.sample(frac=1, random_state=42)

# 3. Формируем X и y
columns = ['NEGATIVE', 'NEUTRAL', 'POSITIVE', 'open',
           'close_prev', 'high_prev', 'low_prev', 'volume_prev_rvol']

X_train = df_train_balanced[columns]
y_train = df_train_balanced['price_increased']

X_test = df_test[columns]  # Тест остался нетронутым!
y_test = df_test['price_increased']

print(f"Обучаем на: {len(X_train)} примерах (сбалансировано)")
print(f"Тестируем на: {len(X_test)} примерах (реальный рынок)")

# 4. Обучаем Random Forest
model = RandomForestClassifier(n_estimators=100,
                               min_samples_leaf=5, # Ограничиваем переобучение
                               n_jobs=-1,
                               random_state=42)
model.fit(X_train, y_train)

# 5. Применяем твою стратегию "Уверенности" (Threshold)
y_probs = model.predict_proba(X_test)[:, 1] # Вероятность роста

Обучаем на: 119600 примерах (сбалансировано)
Тестируем на: 29979 примерах (реальный рынок)


In [ ]:
y_final_pred = []
# Попробуем более мягкий порог, чтобы поймать сделки
threshold_high = 0.63
threshold_low = 0.5

for p in y_probs:
    if p > threshold_high:
        y_final_pred.append(1)
    elif p < threshold_low:
        y_final_pred.append(0)
    else:
        y_final_pred.append(2) # Пропуск хода

y_final_pred = np.array(y_final_pred)

# Фильтруем и считаем метрики
mask = y_final_pred != 2
if mask.sum() > 0:
    acc = accuracy_score(y_test[mask], y_final_pred[mask])
    print(f"\n=== РЕЗУЛЬТАТЫ СТРАТЕГИИ (Порог {threshold_high}/{threshold_low}) ===")
    print(f"Совершено сделок: {mask.sum()} из {len(y_test)} (Coverage: {mask.sum()/len(y_test)*100:.2f}%)")
    print(f"Точность (Win Rate): {acc:.4f}")
    print("\n", classification_report(y_test[mask], y_final_pred[mask]))
else:
    print("Модель не сделала ни одной ставки. Снизь порог уверенности.")


=== РЕЗУЛЬТАТЫ СТРАТЕГИИ (Порог 0.63/0.5) ===
Совершено сделок: 15787 из 29979 (Coverage: 52.66%)
Точность (Win Rate): 0.5138

               precision    recall  f1-score   support

           0       0.51      0.97      0.67      8089
           1       0.52      0.04      0.07      7698

    accuracy                           0.51     15787
   macro avg       0.52      0.50      0.37     15787
weighted avg       0.52      0.51      0.38     15787



примерно за 4 месяца совершено 205 сделок с винрейтом 73,2% процента

In [ ]:
df

,PUBLISHED_ON,PUBLISHED_ON_DATE,NEGATIVE,NEUTRAL,POSITIVE,timestamp,open,high,low,close,volume,price_increased,open_prev,high_prev,low_prev,volume_rvol,close_prev,volume_prev_rvol
21,1698350425,2023-10-26 20:00:25,0.0,1.0,0.0,2023-10-26 20:00:00,0.000909,34059.6,34016.9,34049.0,2.448694,0,0.000279,-0.000109,-0.000012,1.678323,-0.000353,1.028181
22,1698350436,2023-10-26 20:00:36,0.0,1.0,0.0,2023-10-26 20:00:00,0.000000,34059.6,34016.9,34049.0,2.448694,1,0.000909,0.001173,0.000512,1.588071,0.001456,1.678323
23,1698351172,2023-10-26 20:12:52,1.0,0.0,0.0,2023-10-26 20:12:00,0.002532,34135.9,34119.2,34135.9,0.573015,1,0.000000,0.000000,0.000000,0.374260,0.000000,1.588071
24,1698352107,2023-10-26 20:28:27,0.0,0.0,1.0,2023-10-26 20:28:00,0.001204,34189.0,34167.7,34176.1,4.082389,1,0.002532,0.002240,0.003007,2.407545,0.002552,0.374260
25,1698352200,2023-10-26 20:30:00,1.0,0.0,0.0,2023-10-26 20:30:00,0.000299,34197.0,34177.9,34188.8,2.353133,0,0.001204,0.001556,0.001421,1.350726,0.001178,2.407545
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
182495,1763119176,2025-11-14 11:19:36,0.0,0.0,1.0,2025-11-14 11:19:00,-0.002598,96545.2,96421.3,96468.5,6.824971,1,0.000180,-0.000811,-0.001166,1.415614,-0.000933,1.463253
182496,1763119490,2025-11-14 11:24:50,1.0,0.0,0.0,2025-11-14 11:24:00,0.000903,96697.8,96570.0,96660.4,4.755889,0,-0.002598,-0.001954,-0.001891,0.979833,-0.001828,1.415614
182497,1763119800,2025-11-14 11:30:00,1.0,0.0,0.0,2025-11-14 11:30:00,-0.000362,96566.5,96378.7,96378.7,5.635793,0,0.000903,0.001581,0.001542,1.164856,0.001989,0.979833
182498,1763120559,2025-11-14 11:42:39,1.0,0.0,0.0,2025-11-14 11:42:00,-0.004516,96099.0,95965.0,96025.8,5.041239,0,-0.000362,-0.001358,-0.001981,1.074802,-0.002914,1.164856
